## Operational Flow 


In [1]:
import uuid

from sqlglot import parse_one, exp
from sqlglot.expressions import Expression

# import owlready2
from owlready2 import get_ontology, sync_reasoner, sync_reasoner_pellet
# owlready2.set_log_level(1)

### Step-1 : SQL string to AST object

In [2]:
class SqlglotOperator:
    """
    A dedicated class to encapsulate and operate on a sqlglot AST object.

    This operator automatically parses a SQL string, decorates the resulting
    AST with unique IDs for every node, and provides a suite of methods
    for easy and robust inspection and manipulation of the query structure.
    """

    def __init__(self, sql: str):
        """
        Initializes the operator by parsing the SQL and preparing the AST.
        """
        try:
            self.ast: Expression = parse_one(sql)
            self._decorate_ast_with_ids()
            self._map_ids_to_nodes()
        except Exception as e:
            print(f"Error initializing SqlglotOperator: {e}")
            self.ast = None
            self.id_to_node_map = {}

    def _decorate_ast_with_ids(self):
        """
        Private helper to traverse the AST and assign a unique ID to a 'meta'
        attribute on each node. This is fundamental for all ID-based operations.
        """
        if not self.ast:
            return
        for node in self.ast.walk():
            if not hasattr(node, 'meta'):
                node.meta = {}
            node.meta['id'] = "n"+str(uuid.uuid4())[:6]

    def _map_ids_to_nodes(self):
        """
        Private helper to create a dictionary mapping unique IDs to their
        corresponding nodes for efficient lookups.
        """
        if not self.ast:
            self.id_to_node_map = {}
            return
        self.id_to_node_map = {
            node.meta['id']: node for node in self.ast.walk() if hasattr(node, 'meta')
        }
        # print(f"Mapped {len(self.id_to_node_map)} nodes to their IDs.\n Mapped node Ids: {list(self.id_to_node_map.keys())}...")  # Print first 10 IDs for brevity

    def to_sql(self, pretty: bool = True) -> str:
        """
        Generates the SQL string from the current state of the AST.
        """
        if not self.ast:
            return "-- Invalid AST"
        return self.ast.sql(pretty=pretty)

    def get_node_by_id(self, node_id: str) -> Expression | None:
        """
        Retrieves a single node from the AST by its unique ID.
        """
        # print(f"Looking for node with ID: {node_id}")
        # print(f"Total nodes in AST: {len(self.id_to_node_map)} | {self.id_to_node_map.keys()}")
        return self.id_to_node_map.get(node_id)

    def get_nodes_by_type(self, node_type: type) -> list[Expression]:
        """
        Finds all nodes in the AST that are of a specific type.
        """
        if not self.ast:
            return []
        return self.ast.find_all(node_type)

    def remove_node_by_id(self, target_id: str):
        """
        Removes a single node from the AST based on its unique ID.
        The internal AST is updated with the result.
        """
        if not self.ast or not target_id:
            return

        def transformer(node):
            if hasattr(node, 'meta') and node.meta.get('id') == target_id:
                return None  # Returning None deletes the node
            return node

        # transform() returns a new AST object. We must update our instance.
        new_ast = self.ast.transform(transformer)
        self.ast = new_ast
        # After modification, the ID map needs to be rebuilt.
        self.id_to_node_map.pop(target_id, None)  # Remove the ID from the map
        
    def add_node(self, parent_id: str, new_node: Expression, arg_name: str):
        """
        Adds a new node as a child of an existing node.
        """
        parent_node = self.get_node_by_id(parent_id)
        print(f"Adding new node with ID '{new_node}' under parent node '{parent_node}'")
        if not parent_node:
            print(f"[Error] Parent node with ID '{parent_id}' not found.")
            return

        # Decorate the new node and its children with IDs before adding
        # self._decorate_ast_with_ids_on_subtree(new_node)

        # Handle list arguments (like 'expressions' or 'joins')
        if isinstance(getattr(parent_node, arg_name, None), list):
            # FIX: Get the list attribute dynamically using arg_name
            list_arg = getattr(parent_node, arg_name)
            # FIX: Append to the list we just retrieved
            list_arg.append(new_node)
        # Handle single arguments (like 'where' or 'limit')
        else:
            parent_node.set(arg_name, new_node)
        
        # Rebuild the map since we've added new nodes
        # self._map_ids_to_nodes()

    def replace_node(self, target_id: str, new_node: Expression):
        """
        Replaces a target node with a new node.
        """
        if not self.ast or not target_id:
            return
            
        self._decorate_ast_with_ids_on_subtree(new_node)

        def transformer(node):
            if hasattr(node, 'meta') and node.meta.get('id') == target_id:
                return new_node
            return node
            
        self.ast = self.ast.transform(transformer)
        self._map_ids_to_nodes()

    def add_where_condition(self, condition_sql: str):
        """
        Adds a new condition to the main WHERE clause of the query from a string.
        """
        if not self.ast:
            print("[Error] AST is not initialized.")
            return

        # The .where() helper on the main query expression is the easiest way.
        # It handles both creating a new WHERE and AND-ing to an existing one.
        # copy=False ensures the modification happens in-place.
        self.ast.where(condition_sql, copy=False)

        # After modification, the AST structure has changed, so we need to
        # re-decorate the new parts and rebuild the ID map.
        self._decorate_ast_with_ids_on_subtree(self.ast)
        # self._map_ids_to_nodes()

    def _decorate_ast_with_ids_on_subtree(self, ast_node: Expression):
        """Helper to add IDs to a new subtree before it's inserted."""
        for node in ast_node.walk():
            if not hasattr(node, 'meta'):
                node.meta = {}
            if 'id' not in node.meta:
                 node.meta['id'] = str(uuid.uuid4())

    def pretty_print(self):
        """Prints a visual, indented tree of the current AST with node IDs."""
        if not self.ast:
            print("-- Invalid AST")
            return
            
        for node in self.ast.walk():
            indent = "  " * node.depth
            short_id = node.meta.get('id', 'N/A')[:8]
            node_name = node.__class__.__name__
            
            # Add extra detail for common nodes
            if isinstance(node, exp.Identifier):
                node_name += f" ({node.this})"
                continue
            elif isinstance(node, exp.Column):
                 node_name += f" ({node.sql()})"

            print(f"{indent}├── {node_name}  [ID: {short_id}]")

### Step-2 : AST (`sqlgpt_parser`) to AST Tree conversion

In [3]:
class TreeNode:
    """ 
    TreeNode represents a node in the SQL AST tree. It can be a statement, clause, or expression.
    """
    def __init__(self, sqlglot_node, parent=None):
        self.remove = False
        self.parent = parent
        self.children = []
        self.sqlglot_node = sqlglot_node
        self.kind, self.name = ASTTreeOperator.map_node_type(sqlglot_node, statement=True if parent is None else False)

        if self.kind == "Statement":
            self.id = "s" + sqlglot_node.meta['id'][1:]
        else:
            self.id = sqlglot_node.meta['id']
        # sqlglot_node.meta['id'] = self.id

        if self.name == "ColumnRef":
            self.refcol = None
            self.reftable = None
            # self.reference_id = None
            parts = sqlglot_node.parts
            if len(parts) == 2:
                self.reftable = parts[0].this
                self.refcol = parts[1].this
            elif len(parts) == 1:
                self.refcol = parts[0].this

        if self.name == "TableRef":
            self.reftable = sqlglot_node.name
            if sqlglot_node.alias:
                self.refalias = sqlglot_node.alias
            # self.reference_id = None

        if self.name == "Alias":
            alias_expr = sqlglot_node.args.get("alias")
            if isinstance(alias_expr, exp.Identifier):
                self.alias = alias_expr.name

        if self.name == "Literal":
            self.value = sqlglot_node.this
        
        if self.name == "Operator":
            if (type(sqlglot_node).__name__ in ["And", "Or"]):
                self.logics = True 
            else:
                self.logics = False    

    def add_child(self, child):
        self.children.append(child)

    def remove_child(self, child):
        if child in self.children:
            self.children.remove(child)
        else:
            raise ValueError(f"Child {child} not found in children of {self}")

    def __bool__(self):
        # A custom truthiness check: Node is True if it has a non-None value
        return self.id is not None

    def __eq__(self, other):
            if not isinstance(other, TreeNode):
                return NotImplemented  # Or raise TypeError
            return self.id == other.id #and self.kind == other.kind and self.name == other.name

    def __repr__(self):
        suffix = f" ({self.name})"
        if self.name == "ColumnRef":
            suffix += f" [reftable={self.reftable}, refcol={self.refcol}]"
        if self.name == "TableRef":
            suffix += f" [reftable={self.reftable}]"
        if self.name == "Alias" and hasattr(self, "alias"):
            suffix += f" [alias={self.alias}]"
        if self.name == "Literal" and hasattr(self, "value"):
            suffix += f" [value={self.value}]"

        if hasattr(self, "reference_id"):
            suffix += f" [reference_id={self.reference_id}]"
        if hasattr(self, "remove"):
            suffix += f" [remove={self.remove}]"
        if hasattr(self, "refalias"):
            suffix += f" [refalias={self.refalias}]"
        if hasattr(self, "logics"):
            suffix += f" [logics={self.logics}]"
        return f"<({self.id}) {self.kind} | {suffix}>"

    def walk(self):
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)


class ASTTreeOperator:
    """
    Builds and operates on a simplified, custom tree structure derived
    from a sqlglot AST, making it easier to apply domain-specific logic.
    """
    # --- Type Mapper (integrated as a static method) ---
    CLAUSE_TYPES = {
        "From": "FromClause",
        "Group": "GroupByClause",
        "Into": "IntoClause",
        "Limit": "LimitClause",
        "Select": "SelectClause",
        "Set": "SetClause",
        "Update": "UpdateClause",
        "Values": "ValuesClause",
        "Where": "WhereClause",
        "Order": "OrderByClause",
        "Having": "HavingClause",
        "Join": "JoinClause",
        "Insert": "InsertClause",
        "Delete": "DeleteClause"
    }

    STATEMENT_TYPES = {
        "Delete": "DeleteStatement",
        "Insert": "InsertStatement",
        "Update": "UpdateStatement",
        "Select": "SelectStatement"
    }
    
    EXPRESSION_CATEGORIES = {
        "Alias": "Alias", 
        "Table": "TableRef", 
        "Column": "ColumnRef", 
        "Star": "Wildcard",
        "Identifier": "Identifier", 
        "Sum": "Function", "Count": 
        "Function", "Avg": "Function",
        "Max": "Function", 
        "Min": "Function", 
        "And": "Operator", 
        "Or": "Operator", 
        "EQ": "Operator",
        "GT": "Operator", 
        "LT": "Operator", 
        "GTE": "Operator", 
        "LTE": "Operator", 
        "Literal": "Literal"
    }

    @staticmethod
    def map_node_type(node: Expression, statement=False):
        """Maps a sqlglot AST node to a simplified (Kind, Name) tuple."""
        class_name = type(node).__name__
        if statement:
            if class_name in ASTTreeOperator.STATEMENT_TYPES:
                return "Statement", ASTTreeOperator.STATEMENT_TYPES[class_name]
            else:
                return "Statement", class_name
        else:
            if class_name in ASTTreeOperator.CLAUSE_TYPES:
                return "Clause", ASTTreeOperator.CLAUSE_TYPES[class_name]
            elif class_name in ASTTreeOperator.EXPRESSION_CATEGORIES:
                return "Expression", ASTTreeOperator.EXPRESSION_CATEGORIES[class_name]
            # Default fallback
            else:
                return "Expression", class_name

    def __init__(self, sql_operator: SqlglotOperator):
        """
        Initializes the tree operator using a pre-configured SqlglotOperator.
        """
        self.sql_op = sql_operator
        self.root: TreeNode | None = None
        self.id_to_node_map: dict[str, TreeNode] = {}
        if self.sql_op and self.sql_op.ast:
            self.build_tree(self.sql_op.ast)

    def create_column_node(self, parent_sqlglot_node, parent_node, node_id: str, **kwargs) -> 'TreeNode':
        """
        Creates a new sqlglot Column node and wraps it in a TreeNode.
        """
        column_name = kwargs.get("refcol")
        table_name = kwargs.get("reftable")

        if not column_name:
            raise ValueError("A 'this' keyword argument with the column name is required.")

        # 1. Create the Identifier for the column name itself.
        # exp.to_identifier() is a safe way to handle this.
        column_identifier = exp.to_identifier(column_name)
        
        # 2. Prepare the arguments for the main Column expression.
        column_args = {"this": column_identifier}
        
        # 3. If a table name is provided, create its Identifier and add it.
        if table_name:
            table_identifier = exp.to_identifier(table_name)
            column_args["table"] = table_identifier

        # 4. Create the final sqlglot Column node with the prepared arguments.
        new_sqlglot_node = exp.Column(**column_args)

        self.sql_op.add_node(parent_sqlglot_node.meta['id'], new_sqlglot_node, arg_name="expressions")
        
        # 5. Decorate the new node with the provided ID.
        new_sqlglot_node.meta['id'] = node_id
        
        # 6. Wrap the new sqlglot node in your custom TreeNode.
        # The parent is None as it has not been inserted into the main tree yet.
        # create a new TreeNode for the custom tree
        new_tree_node = TreeNode(new_sqlglot_node, parent=parent_node)
        self.id_to_node_map[new_tree_node.id] = new_tree_node
        self.sql_op.id_to_node_map[new_tree_node.id] = new_tree_node
        parent_node.add_child(new_tree_node)
        return new_tree_node
    
    def remove_node_by_id(self, target_id: str):
        """
        Removes a single node from the custom tree based on its unique ID.
        The internal AST is updated with the result.
        """
        if not self.root or not target_id:
            return
        
        # Find the node in the custom tree
        node_to_remove = self.get_node_by_id(target_id)
        if not node_to_remove:
            print(f"[Error] Node with ID '{target_id}' not found.")
            return
        
        # Remove from the parent
        if node_to_remove.parent:
            node_to_remove.parent.remove_child(node_to_remove)

        # Remove from the id map
        del self.id_to_node_map[node_to_remove.id]

        # Also remove from the sqlglot AST
        self.sql_op.remove_node_by_id(target_id)

    def build_tree(self, sqlglot_node, parent=None):
        # print(f"Building tree for node: {repr(sqlglot_node)} | Parent: {parent}")
        if parent is not None:
            clause_root = TreeNode(sqlglot_node, parent=parent)
            self.id_to_node_map[clause_root.id] = clause_root
            root_node = parent
            root_node.add_child(clause_root)
        else:
            root_node = TreeNode(sqlglot_node)
            self.id_to_node_map[root_node.id] = root_node
            self.root = root_node
            clause_root = TreeNode(sqlglot_node, parent=root_node)
            self.id_to_node_map[clause_root.id] = clause_root
            root_node.add_child(clause_root)
        

        found_first_clause = False
        for key, child in sqlglot_node.args.items():
            if child is None:
                continue

            children = child if isinstance(child, list) else [child]

            for expr in children:
                if not isinstance(expr, exp.Expression): # did not understand the logic of the check
                    continue

                c_kind, c_name = ASTTreeOperator.map_node_type(expr)

                if not found_first_clause and c_kind != "Clause":
                    sub = self._build_recursive(expr, clause_root)
                    if sub:
                        clause_root.add_child(sub)
                else:
                    found_first_clause = True
                    clause_node = TreeNode(expr, parent=root_node)
                    self.id_to_node_map[clause_node.id] = clause_node
                    root_node.add_child(clause_node)

                    for _, grandchild in expr.args.items():
                        if isinstance(grandchild, exp.Expression):
                            if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier): # this is a weak check, so try to transfer this is recursion
                                continue  # Skip Identifier under Table
                            child_node = self._build_recursive(grandchild, clause_node)
                            if child_node:
                                clause_node.add_child(child_node)
                        elif isinstance(grandchild, list):
                            for item in grandchild:
                                if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                    continue  # Skip Identifier under Table
                                if isinstance(item, exp.Expression):
                                    child_node = self._build_recursive(item, clause_node)
                                    if child_node:
                                        clause_node.add_child(child_node)
        if parent is None:
            return root_node

    def _build_recursive(self, sqlglot_node: Expression, parent_node: TreeNode = None) -> TreeNode | None:
        if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "ColumnRef":
            return None

        if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "TableRef":
            return None
        
        if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "Alias":
            return None

        if isinstance(sqlglot_node, exp.TableAlias) and parent_node.name == "TableRef":
            return None

        if isinstance(sqlglot_node, exp.Paren):
            # return first child of the paren expression to avoid unnecessary nesting
            first_child = next(iter(sqlglot_node.args.values()))
            return self._build_recursive(first_child, parent_node)

        if isinstance(sqlglot_node, exp.Select):
            current = self.build_tree(sqlglot_node, parent_node)
        else:
            current = TreeNode(sqlglot_node, parent_node)
            self.id_to_node_map[current.id] = current
            for _, child in sqlglot_node.args.items():
                if isinstance(child, exp.Expression):
                    if isinstance(sqlglot_node, exp.Table) and isinstance(child, exp.Identifier):
                        continue
                    child_node = self._build_recursive(child, current)
                    if child_node:
                        current.add_child(child_node)
                elif isinstance(child, list):
                    for item in child:
                        if isinstance(sqlglot_node, exp.Table) and isinstance(item, exp.Identifier):
                            continue
                        if isinstance(item, exp.Expression):
                            child_node = self._build_recursive(item, current)
                            if child_node:
                                current.add_child(child_node)

        return current

    def walk(self):
        """Yields all nodes in the custom tree in pre-order traversal."""
        if not self.root:
            return
        
        nodes_to_visit = [self.root]
        while nodes_to_visit:
            current = nodes_to_visit.pop(0)
            yield current
            nodes_to_visit.extend(current.children)

    def get_node_by_id(self, node_id: str) -> TreeNode | None:
        """Gets a TreeNode from the custom tree by its ID."""
        return self.id_to_node_map.get(node_id)

    def get_parent(self, node_id: str) -> TreeNode | None:
        """Gets the immediate parent of a node."""
        node = self.get_node_by_id(node_id)
        return node.parent if node else None

    def get_parent_clause(self, node_id: str) -> TreeNode | None:
        """Finds the first ancestor of a node that is a 'Clause'."""
        current_node = self.get_node_by_id(node_id)
        while current_node:
            if current_node.kind == "Clause":
                return current_node
            current_node = current_node.parent
        return None

    def get_parent_statement(self, node_id: str) -> TreeNode | None:
        """Finds the first ancestor of a node that is a 'Statement'."""
        current_node = self.get_node_by_id(node_id)
        while current_node:
            if current_node.kind == "Statement" or current_node.name == "Subquery":
                return current_node
            current_node = current_node.parent
        return None

    def get_tables_in_statement(self, stmt_node: TreeNode) -> list[TreeNode]:
        """Finds all TableRef TreeNodes within a given Statement TreeNode."""
        tables = []
        if stmt_node.kind != "Statement":
            return []
        
        from_clause = next((c for c in stmt_node.children if c.name == 'From'), None)
        if from_clause:
            for node in from_clause.children:
                if node.name == "TableRef":
                    tables.append(node)
        return tables
    
    def find_immediate_subqueries(self, node: 'TreeNode') -> list['TreeNode']:
        """
        Finds all Subquery nodes that are descendants of the given `node`
        but not nested inside another subquery within that same scope.
        """
        subqueries = []
        
        def _walker(current_node, is_top_level=True):
            if not is_top_level and current_node.name == 'Subquery':
                subqueries.append(current_node)
                return # Stop descending further

            for child in current_node.children:
                _walker(child, is_top_level=False)

        _walker(node)
        return subqueries

    def get_tables_in_from_clause(self, stmt_node: 'TreeNode') -> list['TreeNode']:
        """
        Finds the FROM clause of a statement and returns all TableRef nodes within it.
        This needs to handle simple FROMs and complex JOINs.
        """
        # This is a simplified example. A full implementation would need
        # to correctly navigate different statement types (SELECT, UPDATE, etc.)
        # and their structures to find the FROM/JOIN clauses.
        from_or_join_nodes = [
            n for n in stmt_node.walk() 
            if n.name in ('FromClause', 'JoinClause', 'UpdateClause', 'InsertClause', 'DeleteClause') and self.get_parent_statement(n.id) == stmt_node
        ]
        # print(f"Found {len(from_or_join_nodes)} FROM/Join nodes in statement {stmt_node.id}.")
        
        table_refs = []
        for node in from_or_join_nodes:
            for child in node.walk():
                if child.name == 'TableRef' and child not in table_refs:
                    table_refs.append(child)
        return table_refs


    def get_select_list(self, stmt_node: 'TreeNode') -> list['TreeNode']:
        """
        Finds the SELECT clause of a statement and returns a list of its
        projection items (e.g., the nodes for columns, aliases, literals).
        """
        # Simplified example:
        for child in stmt_node.children:
            if child.name == 'Select':
                # The children of the SelectNode are the projection items.
                return child.children
        return []

    def pretty_print(self):
        """Prints a visual, indented tree of the custom AST."""
        if not self.root:
            print("-- Tree is not built --")
            return
        
        def print_recursive(node, prefix="", is_last=True):
            connector = "└── " if is_last else "├── "
            print(f"{prefix}{connector}{node}")
            
            for i, child in enumerate(node.children):
                new_prefix = prefix + ("    " if is_last else "│   ")
                print_recursive(child, new_prefix, is_last=(i == len(child.parent.children) - 1))
        
        print_recursive(self.root)


### Step-3 : AST Tree to Ontology Instance 

with Ontology reasoning

In [4]:
class OntologyOperator:
    """Orchestrates the resolution and instantiation of an ontology from an AST tree."""
    def __init__(self, tree_operator: ASTTreeOperator, onto_path: str):
        self.tree_op = tree_operator
        self.onto = get_ontology(onto_path).load()
        self.class_lookup = {cls.name: cls for cls in self.onto.classes()}
        self.created_instances = []
        self.table_ref_instances = []
        self.column_ref_instances = []

        self._create_object_mapper()

    def _create_object_mapper(self):
        """
        Creates a mapping of table and column names to their corresponding
        """
        self.table_entities = {t.TableName[0]: t.name for t in self.get_individuals_by_class("Table")}
        self.column_lookup = {(tbl.TableName[0], c.ColumnName[0]): c.name for c in self.get_individuals_by_class("Column") for tbl in c.columnOfTable}
        

    def get_table_ref_instances(self):
        """
        Returns the list of instantiated TableRef objects.
        """
        mapper = {
            "apuf.RowTag" : "RowTag",
            "apuf.Violated" : "Violated",
        }
        return {inst.name: "Aligned" if getattr(inst, 'hasStatus', []) == [] else mapper.get(str(getattr(inst, 'hasStatus', [])[0])) for inst in self.table_ref_instances }
    
    def get_column_ref_instances(self):
        """
        Returns the list of instantiated ColumnRef objects.
        """
        
        return {inst.name: "Aligned" if getattr(inst, 'hasStatus', []) == [] else "Violated" for inst in self.column_ref_instances }

    def get_individuals_by_class(self, class_name: str) -> list:
        """
        Gets all individuals that are instances of a given class name.
        """
        # Find the actual class object from the lookup map created in __init__
        target_class = self.class_lookup.get(class_name)

        if not target_class:
            print(f"[Warning] Class '{class_name}' not found in the ontology.")
            return []

        # The onto.search() method efficiently finds all individuals of the given type.
        return self.onto.search(type=target_class)
    
    def get_node_attribute(self, node_id: str, attr_name: str) -> str | None:
        """
        Retrieves a specific attribute from an ontology instance (node).
        """
        try:
            node = self.onto[node_id]
            # print all attributes of the node
            # for attr in dir(node):
            #     if not attr.startswith('_') and not callable(getattr(node, attr)):
            #         print(f"node_id: {node_id} | Attribute: {attr} | Value: {getattr(node, attr)}")

            value = getattr(node, attr_name, None)
            
            return str(value) if value is not None else None
        except (KeyError, AttributeError):
            return None

    
    def resolve_wildcard(self):
        """
        Finds and expands all wildcards in the query. This modifies the
        underlying AST and rebuilds the custom tree. Call before instantiation.
        """
        wildcard_nodes = [node for node in self.tree_op.walk() if node.name == "Wildcard"]
        if not wildcard_nodes:
            return

        # print(f"--> Found {len(wildcard_nodes)} wildcard(s). Expanding...")
        for wc_node in wildcard_nodes:
            parent_node = self.tree_op.get_parent(wc_node.id)
            parent_stmt_treenode = self.tree_op.get_parent_statement(wc_node.id)
            if not parent_stmt_treenode: return

            tables_in_stmt = self.tree_op.get_tables_in_from_clause(parent_stmt_treenode)
            if len(tables_in_stmt) != 1:
                print(f"[Warning] Wildcard expansion only supported for single-table queries. Skipping.")
                return

            # print(f"Wildcard node id: {wc_node.id} | Parent node: {parent_node.id} | Parent statement: {parent_stmt_treenode.id}")
            wildcard_sqlglot_node = self.tree_op.sql_op.get_node_by_id(wc_node.id)
            # print(f"Wildcard SQLGlot node: {wildcard_sqlglot_node}")
            parent_select_sqlglot_node = wildcard_sqlglot_node.find_ancestor(exp.Select)
            
            # print(f" tables_in_stmt: {tables_in_stmt}")
            table_name = tables_in_stmt[0].reftable

            table_ind = self.onto.search_one(TableName=table_name)
            if not table_ind or not hasattr(table_ind, "hasColumn"):
                print(f"[Warning] Table '{table_name}' not in ontology or has no columns. Cannot expand.")
                return
            
            # go through all columns of the table and create ColumnRef nodes
            for col in table_ind.hasColumn:
                # print(f"Expanding wildcard for column: {col.ColumnName[0]} in table: {table_name}")
                col_name = col.ColumnName[0]
                new_node_id = "n" + str(uuid.uuid4())[:6]
                # print(f"Column name : {col_name} | New node ID: {new_node_id}, Table name: {table_name}")
                new_col_node = self.tree_op.create_column_node(
                    parent_sqlglot_node=parent_select_sqlglot_node,
                    parent_node=parent_node,
                    node_id=new_node_id,
                    refcol=col_name,
                    reftable=table_name
                )

            # Remove the wildcard node from the parent statement
            self.tree_op.remove_node_by_id(wc_node.id)

    def resolve_references(self, node: 'TreeNode', parent_context={}) -> dict:
        """
        Recursively resolves table and column references in the AST, mapping them to
        database entity IDs.
        """
        # print(f"Resolving references in node: {node.id} | {node.name}")
        subquery_contexts = {}

        in_context_sources = {} 
        from_clause_tables = self.tree_op.get_tables_in_from_clause(node)
        # print(f"Found {len(from_clause_tables)} table(s) in FROM clause.")
        for table_node in from_clause_tables:
            table_name = table_node.reftable
            in_context_sources[table_name] = {"type": "base_table", "name": table_name}
            alias = getattr(table_node, 'refalias', None)
            if alias:
                in_context_sources[alias] = {"type": "base_table", "name": table_name}

        # print(f"Initial in-context sources: {in_context_sources}")

        # --- First Walk (unchanged) ---
        subqueries_in_scope = self.tree_op.find_immediate_subqueries(node)
        for subquery_node in subqueries_in_scope:
            exposed_schema = self.resolve_references(subquery_node, in_context_sources)
            parent = self.tree_op.get_parent(subquery_node.id)
            
            if parent and parent.name == "Alias":
                subquery_contexts[subquery_node.id] = { "alias": parent.alias, "schema": exposed_schema }

        # --- Finalize In-Context References (unchanged) ---
        # add parent context sources to the in_context_sources
        for alias, source in parent_context.items():
            if alias not in in_context_sources:
                in_context_sources[alias] = source

        for sq_id, sq_context in subquery_contexts.items():
            alias = sq_context['alias']
            in_context_sources[alias] = { "type": "subquery", "id": sq_id, "schema": sq_context['schema'] }
        # print(f"Resolved {len(in_context_sources)} in-context sources for node {node.id} | context : {in_context_sources}.")
        # --- Second Walk: UPDATED to use new resolver methods ---
        for current_node in self._walk_and_skip_subqueries(node):
            if current_node.name == 'ColumnRef':
                self._resolve_single_column(current_node, in_context_sources)
            
            elif current_node.name == 'TableRef':
                # Call the new dedicated table resolver
                self._resolve_single_table(current_node, in_context_sources)

        # --- Prepare and Return Exposed Schema (unchanged) ---
        output_schema = {"columns": {}}
        select_list_nodes = self.tree_op.get_select_list(node)
        for item_node in select_list_nodes:
            output_name = None
            resolved_source_info = "expression"
            if hasattr(item_node, 'resolved_source'):
                resolved_source_info = item_node.resolved_source
                output_name = item_node.refcol
            if item_node.name == 'Alias':
                output_name = item_node.alias
                column_expr_node = item_node.children[0]
                if hasattr(column_expr_node, 'resolved_source'):
                    resolved_source_info = column_expr_node.resolved_source
            if output_name:
                output_schema['columns'][output_name] = resolved_source_info

        return output_schema

    # --- NEW: Dedicated method to resolve table references ---
    def _resolve_single_table(self, table_node: 'TreeNode', context_sources: dict):
        """Helper to resolve a single table reference node to its database entity ID."""
        alias = getattr(table_node, 'refalias', table_node.reftable)
        # print(f"Resolving table '{table_node}' | context_sources: {context_sources}")
        if alias in context_sources:
            source_info = context_sources[alias]
            table_node.resolved_info = source_info

            # Only base tables have a direct ID in the database ontology.
            if source_info['type'] == 'base_table':
                real_table_name = source_info['name']
                
                # Use the mapper to get the table's entity ID.
                table_id = self.table_entities.get(real_table_name)
                
                if table_id:
                    # Attach the resolved database ID to the node.
                    table_node.reference_id = table_id
                    # self.table_ref_instances.append(table_node)
                else:
                    raise ValueError(f"Table '{real_table_name}' found in query but not in database entities.")
        else:
            table_id = self.table_entities.get(alias)
            table_node.reference_id = table_id

    # --- UPDATED: Method to resolve column references ---
    def _resolve_single_column(self, col_node: 'TreeNode', context_sources: dict):
        """Helper to resolve a single column reference node to its database entity ID."""
        col_name = col_node.refcol
        table_alias_in_query = col_node.reftable

        # print(f"Resolving column '{col_name}' in context of table alias '{table_alias_in_query}' | context_sources: {context_sources}")

        candidate_sources = []
        if table_alias_in_query:
            if table_alias_in_query in context_sources:
                source = context_sources[table_alias_in_query]
                if self._column_exists_in_source(col_name, source):
                    candidate_sources.append(source)
            else:
                raise NameError(f"Table or alias '{table_alias_in_query}' not found.")
        else:
            # get the base table from the context sources
            for source_alias, source in context_sources.items():
                # Check if the column exists in this source
                if source['type'] == 'base_table':
                    # If the column exists in this base table, add it to candidates
                    if self._column_exists_in_source(col_name, source):
                        candidate_sources.append(source)
                # if self._column_exists_in_source(col_name, source):
                #     candidate_sources.append(source)
        
        if candidate_sources:#len(candidate_sources) == 1:
            source = candidate_sources[0]
            source_alias = next(key for key, val in context_sources.items() if val == source)
            
            col_node.reftable = source_alias
            
            # If the source is a base table, perform the ID lookup.
            if source['type'] == 'base_table':
                real_table_name = source['name']
                col_node.resolved_source = f"{real_table_name}.{col_name}"

                # Use the mapper to get the column's entity ID.
                column_id = self.column_lookup.get((real_table_name, col_name))

                if column_id:
                    # Attach the resolved database ID to the node.
                    col_node.reference_id = column_id
                    # self.column_ref_instances.append(col_node)
                else:
                     raise ValueError(f"Column '{col_name}' in table '{real_table_name}' not in database entities.")
            else: 
                # Source is a subquery; it has no direct database ID.
                subquery_alias = source.get('alias', 'subquery')
                print(f"[Warning] Column '{col_name}' in subquery '{subquery_alias}' has no database ID.")
                col_node.resolved_source = f"{subquery_alias}.{col_name}"
                # self.column_ref_instances.append(col_node)
                
        elif len(candidate_sources) > 1:
            raise ValueError(f"Ambiguous column reference: '{col_name}' exists in multiple tables.")
        else:
            raise ValueError(f"Could not resolve column reference: '{col_name}'.")

    def _column_exists_in_source(self, column_name: str, source_info: dict) -> bool:
        """Checks if a column exists in a given data source (table or subquery)."""
        # print(f"Checking if column '{column_name}' exists in source: {source_info}")
        if source_info['type'] == 'base_table':
            table_name = source_info['name']
            # This is where you would query your ontology.
            # For example, check if `column_name` is a data property of the
            # class corresponding to `table_name`.
            # return self.ontology.is_property_of(column_name, table_name)
            # Placeholder logic:
            # table_class = self.class_lookup.get(table_name)
            return self.column_lookup.get((table_name, column_name), None) is not None
            
        elif source_info['type'] == 'subquery':
            # Check against the exposed schema of the subquery.
            # XXX: check
            return column_name in source_info['schema']['columns']
            
        return False
        
    def _walk_and_skip_subqueries(self, node: 'TreeNode'):
        """
        A generator that yields a node and its descendants. It stops 
        descending into any child that is a 'Subquery'.
        """
        yield node # Yield the current node.

        # Iterate through children and decide whether to recurse.
        for child in node.children:
            # If the child is a subquery, we yield it but do not walk its children.
            # This is because the main `resolve_references` function has already processed
            # it in the first walk.
            if child.name == 'Subquery':
                yield child
            else:
                # If the child is not a subquery, recurse into its subtree.
                yield from self._walk_and_skip_subqueries(child)

    def instantiate_ontology(self, agent_id: str):
        """Walks the tree and creates ontology individuals."""
        print("--> Instantiating ontology individuals...")
        self.created_instances = []
        self.table_ref_instances = []
        self.column_ref_instances = []
        with self.onto:
            self.resolve_wildcard()
            print(self.tree_op.sql_op.ast)
            self.resolve_references(self.tree_op.root)
            
            self._instantiate_recursive(self.tree_op.root, agent_id)
        print(f"    - Instantiated {len(self.table_ref_instances)} TableRefs and {len(self.column_ref_instances)} ColumnRefs.")

    def _instantiate_recursive(self, node: TreeNode, agent_id: str, parent_instance=None, parent=None):
        node_class = self.class_lookup.get(node.name)
        print(f"Instantiating node: {node.id} | Class: {node_class} | Parent: {parent_instance if parent_instance else None}")
        if not node_class: return
        
        inst = node_class(node.id)
        self.created_instances.append(inst)
        if parent_instance: 
            if (parent.kind == "Statement") and (node.kind == "Clause"):
                parent_instance.hasClause.append(inst) 
            elif (parent.kind == "Clause") and (node.kind == "Expression"):
                parent_instance.hasExpression.append(inst)
            else:
                parent_instance.immediateChildNode.append(inst)

        if node.kind == "Statement":
            agent_ref = self.onto.search_one(iri=f"*{agent_id}")
            if agent_ref: inst.executedBy.append(agent_ref)
        
        if hasattr(node, "reference_id") and node.reference_id:
            ref_obj = self.onto.search_one(iri=f"*{node.reference_id}")
            if ref_obj:
                if node.name == "TableRef":
                    inst.referencesToTable.append(ref_obj)
                    self.table_ref_instances.append(inst)
                elif node.name == "ColumnRef":
                    inst.referencesToColumn.append(ref_obj)
                    self.column_ref_instances.append(inst)

        for child in node.children:
            self._instantiate_recursive(child, agent_id, parent_instance=inst, parent=node)

    def reason_and_save(self, output_path: str):
        """Runs the reasoner and saves the final ontology."""
        print("--> Running reasoner and saving ontology...")
        with self.onto:
            sync_reasoner_pellet(infer_property_values = True)
        self.onto.save(file=output_path, format="rdfxml")
        print(f"    - Ontology saved to {output_path}")


In [5]:
class ASTPruner:
    """
    Prunes a sqlglot AST based on the state of an associated ontology.
    """
    base_clause = {
            "SelectStatement": "SelectClause",
            "UpdateStatement": "UpdateClause",
            "InsertStatement": "InsertClause",
            "DeleteStatement": "DeleteClause",
        }

    def __init__(self, onto_op: OntologyOperator):
        self.onto_op = onto_op
        self.sql_op = onto_op.tree_op.sql_op
        self.tree_op = onto_op.tree_op
        self.violated_ids = set()

        

    def prune(self):
        """
        Orchestrates the full, multi-stage pruning process.
        The underlying AST is modified in place.
        """
        print("--- Starting AST Pruning Process ---")
        print(f"Query before pruning: {self.sql_op.ast}")

        self.column_ref_instances = self.onto_op.get_column_ref_instances()
        self.table_ref_instances = self.onto_op.get_table_ref_instances()

        self._prune_from_table_status()
        if self.sql_op.ast is not None:
            # If the AST is None, it means the root statement was violated.
            # No need to proceed further.
            # print(f"    - SQL AST after Stage 1: {self.sql_op.ast}")
            self._prune_from_column_status()
        
        # After all modifications, rebuild the custom tree to reflect changes
        # print("--> Pruning complete. Rebuilding custom tree...")
        # self.tree_op.build_tree()
        # print("--- Pruning Process Finished ---")

    def _prune_from_table_status(self):
        """
        Stage 1: Handles removals and modifications based on TableRef statuses.
        """
        print("--> Stage 1: Pruning based on table statuses...")
        self.violated_ids = []

        for inst, status in self.table_ref_instances.items():

            # Condition A.1: Violated Table -> Remove Statement
            if status == "Violated":
                parent_stmt = self.tree_op.get_parent_statement(inst)
                if parent_stmt:
                    if parent_stmt.id == self.tree_op.root.id:
                        self.sql_op.ast = None
                        # print(f"    - Root statement '{parent_stmt.id[:8]}' is violated. Removing entire AST.")
                        return  # No need to continue, the entire AST is removed.
                    # print(f"    - TableRef '{inst}' is Violated. Marking statement '{parent_stmt.id[:8]}' for removal.")
                    base_clause_type = ASTPruner.base_clause.get(parent_stmt.name, None)
                    clause_id = next((c.id for c in parent_stmt.children if c.name == base_clause_type), None)
                    # print(f"    - Base clause ID: {clause_id}")
                    if clause_id:
                        self.violated_ids.append(clause_id)

            # Condition A.2: RowTag Table -> Add WHERE condition
            elif status == "RowTag":
                # print(f"    - TableRef '{inst}' is RowTag. Adding condition to parent statement.")
                policy_list = self.onto_op.get_node_attribute(inst, "relatedPolicy")
                # print(f"    - Found related policies: {policy_list}")
                if not policy_list: continue
                
                # get data property 'hasCondition' from the first policy
                policy_id = str(policy_list).strip("[]").split(".")[-1]
                # print(f"    - Using policy ID: {policy_id}, {policy_list[0]}")
                condition_list = self.onto_op.get_node_attribute(policy_id, "hasActionCondition")
                # print(f"    - Found conditions: {condition_list}")
                if not condition_list: continue
                
                condition_str = str(condition_list).strip("['']")
                parent_stmt = self.tree_op.get_parent_statement(inst)
                # print(f"parent statement: {parent_stmt}")
                if parent_stmt and parent_stmt.name == "SelectStatement":
                    # print(f"    - TableRef '{inst}' has RowTag. Adding condition to statement '{parent_stmt.id[:8]}': {condition_str}")
                    parent_id = "n" + parent_stmt.id[1:]
                    stmt_sqlglot_node = self.sql_op.get_node_by_id(parent_id)
                    # print(f"    - SQLGlot node for statement: {stmt_sqlglot_node} | {parent_stmt.id}")
                    if stmt_sqlglot_node:
                        stmt_sqlglot_node.where(condition_str, copy=False)
        # try:
            # Step 3: Apply the main pruning transformer with all collected IDs
        self.sql_op.ast = self.sql_op.ast.transform(self._pruning_transformer)
        # except Exception as e:
        #     print(f"[Error] Failed to apply pruning transformer: {e}")
        #     return
        # self.sql_op.ast = self.sql_op.ast.transform(self._pruning_transformer)

    def _prune_from_column_status(self):
        """
        Stage 2: Handles cascading removals based on ColumnRef 'Violated' status.
        This runs iteratively to ensure all cascading effects are resolved.
        """
        print("--> Stage 2: Pruning based on column statuses...")
        # Step 1: Collect initial violated IDs and propagate them
        self.violated_ids = [id for id, status in self.column_ref_instances.items() if status == "Violated"]
        if not self.violated_ids:
            print("     - No violated columns found.")
            return
        
        # print(f"     - Initial violated column IDs: {self.violated_ids}")
        self._propagate_violations_through_aliases()

        # Step 2: Iteratively apply the pruning transformer until the AST is stable
        max_iterations = 10  # Safeguard against potential infinite loops
        for i in range(max_iterations):
            previous_sql = self.sql_op.ast.sql(pretty=True)
            
            # Apply the main pruning transformer
            # print(f"     - Iteration {i + 1}: Applying pruning transformer... {self.sql_op.ast}")
            # try:
            self.sql_op.ast = self.sql_op.ast.transform(self._pruning_transformer)
            # except AssertionError as e:
            #     self.sql_op.ast = None
            #     return

            current_sql = self.sql_op.ast.sql(pretty=True)

            # If the AST has not changed, the pruning is complete
            if current_sql == previous_sql:
                print(f"     - Pruning complete. AST stabilized after {i + 1} iteration(s).")
                break
            
            if i == max_iterations - 1:
                print("     - Warning: Pruning reached max iterations. The AST may not be stable.")
        else:
            # This part runs if the loop finishes without a 'break'
            # which indicates it hit the max_iterations limit.
            print("     - Pruning complete (max iterations reached).")

    def _remove_node_by_id(self, target_id: str):
        """
        Removes a node from the SQLGlot AST and the custom tree by its ID.
        This is a utility method to encapsulate the removal logic.
        """
        # print(f"Removing node with ID: {target_id}")
        
        # Remove from the custom tree
        self.tree_op.remove_node_by_id(target_id)

        # Also remove from the SQLGlot AST
        self.sql_op.remove_node_by_id(target_id)

    def _propagate_violations_through_aliases(self):
        """Finds columns that are aliased and marks their usages as violated."""
        # This is a simplified propagation. A full implementation for complex
        # nested queries would require more sophisticated scope analysis.
        tainted_aliases = set()
        for alias_node in self.sql_op.ast.find_all(exp.Alias):
            source_col_id = alias_node.this.meta.get('id')
            if source_col_id in self.violated_ids:
                tainted_aliases.add(alias_node.alias)
        
        if tainted_aliases:
            # print(f"    - Propagating violations from tainted aliases: {tainted_aliases}")
            for col_node in self.sql_op.ast.find_all(exp.Column):
                if col_node.this.name in tainted_aliases:
                    self.violated_ids.add(col_node.meta.get('id'))

    def _pruning_transformer(self, node):
        """
        The core transformation logic for cascading removals, using the correct
        sqlglot expression classes.
        """
        # Base Case: If the node's ID is marked for removal, remove it.
        if hasattr(node, 'meta') and node.meta.get('id') in self.violated_ids:
            return None

        # Rule 1: Handle logical connectors (AND, OR).
        # This is the corrected class name.
        if isinstance(node, exp.Connector):
            left_is_gone = node.left is None
            right_is_gone = node.right is None

            if left_is_gone and not right_is_gone:
                return node.right  # Promote the survivor
            if right_is_gone and not left_is_gone:
                return node.left   # Promote the survivor
            if left_is_gone and right_is_gone:
                return None        # Both sides gone, remove the operator

        # Rule 2: Handle all other binary operations (Comparisons, Arithmetic).
        elif isinstance(node, exp.Binary):
            if node.left is None or node.right is None:
                return None

        # Rule 3: Clean up a WHERE clause that is now empty.
        if isinstance(node, exp.Where) and node.this is None:
            return None

        # Rule 4: Clean up a SELECT statement that has no columns left.
        if isinstance(node, exp.Select) and not node.expressions:
            # print("     - Cascading removal of SELECT statement with no columns.")
            return None

        # Rule 5: Other clean-up rules.
        if isinstance(node, exp.Join) and node.on is None:
            return None

        return node

---
## Run

In [ ]:
# testing p001 
query = "SELECT * FROM District; "

# testing p002
query = "INSERT INTO Loan (loan_id, account_id, date, amount, duration, payments, status) VALUES (5317, 2, '930712', 96396, 12, 8033, 'B'); "

# # testing p003
query = "UPDATE Account SET frequency = 'WEEKLY' WHERE account_id = 1; "

# testing p004
query = "DELETE FROM Order WHERE order_id = 1; "

# testing p005
query = "SELECT * FROM Transaction WHERE k_symbol = 'UROK'; "

# testing p006 
query = "SELECT account_id, date FROM Transaction; "

# # testing p007 
query = "SELECT client_id, birth_date FROM Client WHERE gender = 'male'; "

In [7]:
sql_op = SqlglotOperator(query)
# view the sqlglot AST
print("SQLGlot AST:")
# sql_op.pretty_print()

SQLGlot AST:


In [8]:
sql_op.ast

Select(
  expressions=[
    Star()],
  from=From(
    this=Table(
      this=Identifier(this=Transaction, quoted=False))),
  where=Where(
    this=EQ(
      this=Column(
        this=Identifier(this=k_symbol, quoted=False)),
      expression=Literal(this='UROK', is_string=True))))

In [9]:
tree_op = ASTTreeOperator(sql_op)
print("\nCustom AST Tree:")
tree_op.pretty_print()


Custom AST Tree:
└── <(se7bdb2) Statement |  (SelectStatement) [remove=False]>
    ├── <(ne7bdb2) Clause |  (SelectClause) [remove=False]>
    │   └── <(n43edc0) Expression |  (Wildcard) [remove=False]>
    ├── <(n4a35b7) Clause |  (FromClause) [remove=False]>
    │   └── <(nabbc30) Expression |  (TableRef) [reftable=Transaction] [remove=False]>
    └── <(n93f654) Clause |  (WhereClause) [remove=False]>
        └── <(n1deebc) Expression |  (Operator) [remove=False] [logics=False]>
            ├── <(ndbdb50) Expression |  (ColumnRef) [reftable=None, refcol=k_symbol] [remove=False]>
            └── <(nf29450) Expression |  (Literal) [value=UROK] [remove=False]>


In [10]:
onto_path = "../ontology_file/aputv4.rdf"
output_path = "../ontology_file/resolved_query_instances.rdf"
agent_id = "a0012"

# 1. Create an instance of the class
instantiator = OntologyOperator(tree_op, onto_path)

# 2. Instantiate the ontology
# 2.a resolve wildcards in the query
# 2.b resolve references in the tree
# 2.c instantiate the ontology individuals
instantiator.instantiate_ontology(agent_id)
tree_op.pretty_print()
# 3. Run the reasoner and save the ontology
instantiator.reason_and_save(output_path)

--> Instantiating ontology individuals...
Adding new node with ID 'Transaction.account_id' under parent node 'SELECT * FROM Transaction WHERE k_symbol = 'UROK''
Adding new node with ID 'Transaction.k_symbol' under parent node 'SELECT *, Transaction.account_id FROM Transaction WHERE k_symbol = 'UROK''
Adding new node with ID 'Transaction.date' under parent node 'SELECT *, Transaction.account_id, Transaction.k_symbol FROM Transaction WHERE k_symbol = 'UROK''
Adding new node with ID 'Transaction.type' under parent node 'SELECT *, Transaction.account_id, Transaction.k_symbol, Transaction.date FROM Transaction WHERE k_symbol = 'UROK''
Adding new node with ID 'Transaction.amount' under parent node 'SELECT *, Transaction.account_id, Transaction.k_symbol, Transaction.date, Transaction.type FROM Transaction WHERE k_symbol = 'UROK''
Adding new node with ID 'Transaction.trans_id' under parent node 'SELECT *, Transaction.account_id, Transaction.k_symbol, Transaction.date, Transaction.type, Transac

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\jena-arq-2.10.0.jar;c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\jena-core-2.10.0.jar;c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\owlready2\pellet\jena-iri-0.9

* Owlready * Adding relation aputv4.n4a35b7 hasParentNode aputv4.se7bdb2
* Owlready * Adding relation aputv4.n4a35b7 immediateParentNode aputv4.se7bdb2
* Owlready * Adding relation aputv4.n4a35b7 hasChildNode aputv4.nabbc30
* Owlready * Adding relation aputv4.n4a35b7 immediateChildNode aputv4.nabbc30
* Owlready * Adding relation aputv4.n7a0fe5 hasParentNode aputv4.ne7bdb2
* Owlready * Adding relation aputv4.n7a0fe5 hasParentNode aputv4.se7bdb2
* Owlready * Adding relation aputv4.n7a0fe5 immediateParentNode aputv4.ne7bdb2
* Owlready * Adding relation aputv4.ne21619 hasParentNode aputv4.ne7bdb2
* Owlready * Adding relation aputv4.ne21619 hasParentNode aputv4.se7bdb2
* Owlready * Adding relation aputv4.ne21619 immediateParentNode aputv4.ne7bdb2
* Owlready * Adding relation aputv4.naa2d37 hasParentNode aputv4.ne7bdb2
* Owlready * Adding relation aputv4.naa2d37 hasParentNode aputv4.se7bdb2
* Owlready * Adding relation aputv4.naa2d37 immediateParentNode aputv4.ne7bdb2
* Owlready * Adding rel

* Owlready2 * Pellet took 4.660593748092651 seconds
* Owlready * Reparenting apuf.c028: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c031: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c016: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c001: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c021: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c011: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c003: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c007: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c024: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c020: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c012: {apuf.Column, apuf.Object} => {apuf.Column}
* Owlready * Reparenting apuf.c017: {apuf.Column, apuf.Object} => {a

In [11]:
print("Column reference node status: ", instantiator.get_column_ref_instances())
print("Table reference node status: ", instantiator.get_table_ref_instances())

Column reference node status:  {'n7a0fe5': 'Violated', 'n0a0a82': 'Violated', 'ne21619': 'Violated', 'nc07b71': 'Violated', 'nb39cf8': 'Violated', 'n28081f': 'Aligned', 'nc97c90': 'Aligned', 'naa2d37': 'Aligned', 'nad98fe': 'Aligned', 'n8965b6': 'Aligned', 'ndbdb50': 'Violated'}
Table reference node status:  {'nabbc30': 'Aligned'}


In [12]:
pruner = ASTPruner(instantiator)

# # 4. Prune the AST based on the ontology state
pruner.prune()

print(sql_op.to_sql())

--- Starting AST Pruning Process ---
Query before pruning: SELECT Transaction.account_id, Transaction.k_symbol, Transaction.date, Transaction.type, Transaction.amount, Transaction.trans_id, Transaction.operation, Transaction.balance, Transaction.bank, Transaction.account FROM Transaction WHERE k_symbol = 'UROK'
--> Stage 1: Pruning based on table statuses...
--> Stage 2: Pruning based on column statuses...
     - Pruning complete. AST stabilized after 4 iteration(s).
SELECT
  Transaction.trans_id,
  Transaction.operation,
  Transaction.balance,
  Transaction.bank,
  Transaction.account
FROM Transaction
